In [2]:
import pandas as pd
import numpy as np
import folium
from folium.plugins import MarkerCluster

# 1) Les commandes préparées dans 01_exploration
commandes = pd.read_pickle("data/commandes.pkl")
print(len(commandes), "commandes")

# 2) Une ligne par ÉTAT du client : position = centre de ses clients
etats = commandes.groupby("customer_state").agg(
    region=("region_client", "first"),
    lat=("lat_client", "mean"),
    lon=("lon_client", "mean"),
    commandes=("order_id", "size"),
    taux_retard=("en_retard", "mean"),
    delai=("delai_livraison_j", "mean"),
    note=("review_score", "mean"),
    ca=("montant", "sum"))
etats["taux_retard"] = etats["taux_retard"] * 100

# 3) Tranches de retard (comme les tranches de facture du TP2)
SEUILS   = [0, 5, 10, 100]
TRANCHES = ["Moins de 5 %", "5 à 10 %", "Plus de 10 %"]
etats["tranche"] = pd.cut(etats["taux_retard"], bins=SEUILS, labels=TRANCHES)

# 4) Une ligne par VENDEUR (pour la couche des vendeurs)
vendeurs = commandes.dropna(subset=["lat_vendeur"]).groupby("seller_id").agg(
    etat=("seller_state", "first"),
    lat=("lat_vendeur", "first"),
    lon=("lon_vendeur", "first"),
    commandes=("order_id", "size"),
    taux_retard=("en_retard", "mean"),
    preparation=("delai_preparation_j", "mean"))
vendeurs["taux_retard"] = vendeurs["taux_retard"] * 100

print(len(etats), "états,", len(vendeurs), "vendeurs")
print(etats["tranche"].value_counts())
etats.sort_values("taux_retard", ascending=False).head(8).round(1)

96018 commandes
27 états, 2926 vendeurs
tranche
Plus de 10 %    11
5 à 10 %         9
Moins de 5 %     7
Name: count, dtype: int64


,region,lat,lon,commandes,taux_retard,delai,note,ca,tranche
customer_state,,,,,,,,,
AL,Nord-Est,-9.6,-36.0,396,21.5,24.5,3.8,94095.9,Plus de 10 %
MA,Nord-Est,-3.6,-44.7,710,17.6,21.6,3.8,146551.9,Plus de 10 %
SE,Nord-Est,-10.9,-37.2,331,15.4,21.5,3.9,69841.9,Plus de 10 %
PI,Nord-Est,-5.5,-42.6,472,14.0,19.5,4.0,104507.1,Plus de 10 %
CE,Nord-Est,-4.3,-39.0,1272,13.8,21.2,3.9,264470.8,Plus de 10 %
RR,Nord,2.8,-60.7,40,12.5,29.9,3.9,8970.5,Plus de 10 %
RJ,Sud-Est,-22.8,-43.2,12291,12.2,15.3,4.0,2044218.7,Plus de 10 %
BA,Nord-Est,-13.0,-39.5,3249,12.2,19.3,3.9,590018.7,Plus de 10 %


In [3]:
# ÉTAPE 2 : la carte de base – un cercle par état du client

# Couleur selon la tranche de retard (une seule couleur d'accent : le rouge = problème)
COULEURS_TRANCHE = {"Moins de 5 %": "#9CA3AF", "5 à 10 %": "#E8A598", "Plus de 10 %": "#B4483F"}

NOMS_ETATS = {"AC": "Acre", "AL": "Alagoas", "AM": "Amazonas", "AP": "Amapá", "BA": "Bahia", "CE": "Ceará",
              "DF": "Distrito Federal", "ES": "Espírito Santo", "GO": "Goiás", "MA": "Maranhão",
              "MG": "Minas Gerais", "MS": "Mato Grosso do Sul", "MT": "Mato Grosso", "PA": "Pará",
              "PB": "Paraíba", "PE": "Pernambuco", "PI": "Piauí", "PR": "Paraná", "RJ": "Rio de Janeiro",
              "RN": "Rio Grande do Norte", "RO": "Rondônia", "RR": "Roraima", "RS": "Rio Grande do Sul",
              "SC": "Santa Catarina", "SE": "Sergipe", "SP": "São Paulo", "TO": "Tocantins"}

def fr(x, dec=0):
    """Format français : espace pour les milliers, virgule pour les décimales."""
    return f"{x:,.{dec}f}".replace(",", " ").replace(".", ",")

def fiche(code, e):
    """Fiche HTML affichée au clic sur un état (popup), comme au TP2."""
    couleur = COULEURS_TRANCHE[e["tranche"]]
    return f"""
    <div style="font-family: Arial; font-size: 13px; min-width: 210px;">
        <b style="font-size: 15px;">{NOMS_ETATS[code]} ({code})</b><br>
        <span style="color: #6B7280;">{e['region']}</span>
        <hr style="margin: 6px 0;">
        Commandes : <b>{fr(e['commandes'])}</b><br>
        En retard : <b style="color: {couleur};">{fr(e['taux_retard'], 1)} %</b><br>
        Délai moyen : <b>{fr(e['delai'], 1)} jours</b><br>
        Note moyenne : <b>{fr(e['note'], 1)} / 5</b><br>
        Chiffre d'affaires : <b>{fr(e['ca'] / 1e6, 2)} M R$</b>
    </div>
    """

# La carte, centrée sur le Brésil
carte = folium.Map(location=[-14, -52], zoom_start=4)

for code, e in etats.iterrows():
    folium.CircleMarker(
        location=[e["lat"], e["lon"]],
        radius=6 + 28 * np.sqrt(e["commandes"] / etats["commandes"].max()),   # taille = nombre de commandes
        color="white", weight=1.5,
        fill=True, fill_color=COULEURS_TRANCHE[e["tranche"]], fill_opacity=0.85,
        tooltip=f"{code} : {fr(e['taux_retard'], 1)} % en retard",
        popup=folium.Popup(fiche(code, e), max_width=280),
    ).add_to(carte)

carte

In [5]:
# ÉTAPE 3 : filtres – une couche par région + une couche des vendeurs

REGIONS = ["Sud-Est", "Sud", "Centre-Ouest", "Nord-Est", "Nord"]
carte = folium.Map(location=[-14, -52], zoom_start=4)

# 1) Une couche (FeatureGroup) par région : on peut cocher / décocher chaque région
for region in REGIONS:
    groupe = folium.FeatureGroup(name=f"Clients – {region}").add_to(carte)
    for code, e in etats[etats["region"] == region].iterrows():
        folium.CircleMarker(
            location=[e["lat"], e["lon"]],
            radius=6 + 28 * np.sqrt(e["commandes"] / etats["commandes"].max()),
            color="white", weight=1.5,
            fill=True, fill_color=COULEURS_TRANCHE[e["tranche"]], fill_opacity=0.85,
            tooltip=f"{code} : {fr(e['taux_retard'], 1)} % en retard",
            popup=folium.Popup(fiche(code, e), max_width=280),
        ).add_to(groupe)

# 2) La couche des vendeurs : 2 926 points regroupés (MarkerCluster), cachée au départ
groupe_vendeurs = folium.FeatureGroup(name="Vendeurs (d'où partent les colis)", show=False).add_to(carte)
cluster = MarkerCluster(options={"disableClusteringAtZoom": 10}).add_to(groupe_vendeurs)
for vid, v in vendeurs.iterrows():
    folium.CircleMarker(
        location=[v["lat"], v["lon"]], radius=4,
        color="#1B6A8F", weight=1, fill=True, fill_color="#1B6A8F", fill_opacity=0.7,
        tooltip=f"Vendeur ({v['etat']}) : {fr(v['commandes'])} commandes, préparation {fr(v['preparation'], 1)} j",
    ).add_to(cluster)

# 3) Le menu des couches (en haut à droite)
folium.LayerControl(collapsed=False).add_to(carte)
# 4) Légende compacte (en bas à droite, comme au TP2)
lignes_tranches = "".join(
    f"<div><span style='display:inline-block;width:11px;height:11px;border-radius:50%;"
    f"background:{couleur};margin-right:6px;'></span>{tranche}</div>"
    for tranche, couleur in COULEURS_TRANCHE.items())

legende = f"""
<div style="position: fixed; bottom: 25px; right: 10px; z-index: 9999; max-width: 230px;
            background: rgba(255,255,255,0.92); padding: 8px 10px; border-radius: 6px;
            box-shadow: 0 1px 4px rgba(0,0,0,0.3); font-family: Arial; font-size: 11px; line-height: 1.5;">
  <b>Commandes en retard (cercle)</b>{lignes_tranches}
  <div style="color:#6B7280;">Taille du cercle = nombre de commandes</div>
  <div style="margin-top:4px;"><span style="display:inline-block;width:8px;height:8px;border-radius:50%;
       background:#1B6A8F;margin-right:6px;"></span>Vendeur (couche à cocher)</div>
  <div style="margin-top:4px; color:#6B7280;">Position = centre des clients de l'état.</div>
</div>
"""
carte.get_root().html.add_child(folium.Element(legende))
carte